# CRAFT-GC — Full Colab Pipeline

## ⚠️ READ FIRST
If you ever see **`getcwd: cannot access parent directories`** or **`FileNotFoundError: /content/Research`**:
1. **Runtime → Restart session** (mandatory — do not skip)
2. **Runtime → Change runtime type → T4 GPU**
3. Run **Cell 1 only** — wait for `SETUP OK`

**HF token:** Colab sidebar 🔑 Secrets → name **`HF_TOKEN`** → Notebook access **ON**  
Or type when Cell 1 prompts you.

| Cell | Task | Time |
|------|------|------|
| 1 | Setup + clone | ~5 min |
| 2 | Smoke test | ~5 min |
| 3 | Stage A (optional) | ~2 min |
| 4 | Stage B (main) | **2–4 hours** |
| 5 | Download zip | ~1 min |

In [ ]:
# === CELL 1: SETUP — run once after each Runtime Restart ===
import os, sys, shutil, subprocess
from getpass import getpass

# CRITICAL: fix broken cwd from previous failed runs
os.chdir("/content")
print("Starting cwd:", os.getcwd())

REPO = "https://github.com/Natiabay/Dynamic-Open-Set-Post-Processing-for-Intersectional-Fairness-in-Text-to-Image-Diffusion-Models.git"
PROJECT = "/content/Research"

# 1) Install packages (always from /content)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "torch", "torchvision", "open-clip-torch", "diffusers", "transformers",
    "accelerate", "pandas", "matplotlib", "seaborn", "huggingface_hub",
], check=True, cwd="/content")

# 2) Hugging Face login
from huggingface_hub import login

def get_hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        print("Secret HF_TOKEN not found — type token below (hidden):")
        return getpass("Hugging Face token: ")

login(token=get_hf_token())

# 3) Delete old folder safely (Python, not shell — avoids getcwd bug)
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
    print("Removed old", PROJECT)

# 4) Clone fresh repo
print("Cloning repository...")
r = subprocess.run(["git", "clone", REPO, PROJECT], cwd="/content", capture_output=True, text=True)
if r.returncode != 0:
    print(r.stdout)
    print(r.stderr)
    raise RuntimeError("git clone failed — Runtime → Restart session, then re-run Cell 1")

if not os.path.isfile(f"{PROJECT}/scripts/colab_run_stage_b.py"):
    raise FileNotFoundError("Clone incomplete — push latest code from PC: git push origin main")

# 5) Install craft_gc package
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", PROJECT], check=True, cwd=PROJECT)

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

import torch
import craft_gc

if not torch.cuda.is_available():
    raise RuntimeError("No GPU! Runtime → T4 GPU → Save → Restart → re-run Cell 1")

print("=" * 55)
print("SETUP OK")
print("Project:", PROJECT)
print("GPU:", torch.cuda.get_device_name(0))
print("craft_gc:", craft_gc.__version__)
print("Script:", os.path.isfile("scripts/colab_run_stage_b.py"))
print("=" * 55)

In [ ]:
# === CELL 2: SMOKE TEST (2 prompts, ~5 min) ===
import subprocess, sys, os

PROJECT = "/content/Research"
assert os.path.isfile(f"{PROJECT}/scripts/colab_run_stage_b.py"), "Run Cell 1 first"

print("Smoke test: 2 prompts × 4 methods × 1 seed = 8 images")
r = subprocess.run(
    [sys.executable, "scripts/colab_run_stage_b.py", "--device", "cuda", "--limit", "2", "--seeds", "42"],
    cwd=PROJECT,
    capture_output=True,
    text=True,
)
print(r.stdout)
if r.stderr:
    print("STDERR:", r.stderr[-4000:])
if r.returncode != 0:
    raise RuntimeError(f"Smoke test failed (exit {r.returncode})")
print("\n✅ SMOKE TEST PASSED — now run Cell 4 for full Stage B")

In [ ]:
# === CELL 3: STAGE A — CLIP embeddings (~2 min, optional) ===
import os
os.chdir("/content/Research")
!python scripts/evaluate_embeddings.py --device cuda
!python scripts/merge_paper_results.py
!python scripts/plot_figures.py
print("Stage A done.")

In [ ]:
# === CELL 4: STAGE B — 1000 images, 2–4 HOURS (do not close tab) ===
import subprocess, sys, os

PROJECT = "/content/Research"
subprocess.run(["git", "-C", PROJECT, "pull"], check=False)

print("Stage B starting: 50 prompts × 4 methods × 5 seeds")
r = subprocess.run(
    [sys.executable, "scripts/colab_run_stage_b.py", "--device", "cuda",
     "--limit", "50", "--seeds", "42", "123", "456", "789", "1024"],
    cwd=PROJECT,
)
if r.returncode != 0:
    raise RuntimeError(f"Stage B failed (exit {r.returncode})")
print("\n✅ STAGE B COMPLETE — run Cell 5 to download")

In [ ]:
# === CELL 5: Download results ===
import os, shutil
from google.colab import files

assert os.path.isdir("/content/Research/results/pilot_images"), "Run Stage B first"
shutil.make_archive("/content/craft_gc_results", "zip", "/content/Research/results")
files.download("/content/craft_gc_results.zip")
print("Downloaded. On PC: unzip into results/ then run python scripts/update_image_table.py")